In [1]:
import ipywidgets as widgets
from ipyleaflet import Map, Marker, AntPath, CircleMarker, basemaps
from IPython.display import display, HTML, clear_output
import time

# --- 1. TỔNG HỢP CSS (Căn chỉnh màn hình đăng nhập) ---
display(HTML("""
<style>
    .phone-frame {
        border: 10px solid #222; border-radius: 40px;
        width: 380px; height: 820px; margin: auto;
        overflow: hidden; background: white;
        box-shadow: 0 20px 50px rgba(0,0,0,0.3);
        display: flex; flex-direction: column;
        font-family: 'Segoe UI', sans-serif; position: relative;
    }
    /* Căn giữa màn hình đăng nhập theo chiều ngang và dọc */
    .screen-content {
        padding: 30px;
        display: flex;
        flex-direction: column;
        align-items: center;
        justify-content: center;
        height: 100%;
        box-sizing: border-box;
    }
    .logo-text { color: #0984e3; font-size: 45px; font-weight: bold; margin-bottom: 40px; letter-spacing: 2px; }
    .widget-text input, .widget-password input {
        background: #f1f2f6 !important; border: 2px solid #ced6e0 !important;
        border-radius: 12px !important; padding: 12px !important; font-weight: 600 !important; color: #2f3542 !important;
    }
    .app-header { background: #00d2d3; color: #1a1a1a; padding: 20px; text-align: center; font-weight: bold; font-size: 16px; letter-spacing: 1px; }
    .app-content { padding: 15px; height: 680px; color: #eee; background: #1a1a1a; overflow-y: hidden; }
    .price-panel { background: #2d3436; padding: 12px; border-radius: 12px; margin-top: 10px; border-left: 5px solid #00d2d3; }
    .primary-btn, .order-btn { width: 100% !important; background: #00d2d3 !important; color: #1a1a1a !important;
                 font-weight: bold !important; border-radius: 20px !important; height: 45px !important; border: none !important; margin-top: 15px; cursor: pointer; }
    .secondary-btn { width: 100% !important; background: transparent !important; color: #0984e3 !important; font-size: 14px !important;
                     margin-top: 15px !important; border: none !important; text-decoration: underline !important; cursor: pointer; font-weight: bold !important; }
    .widget-label-custom { color: #00d2d3; font-size: 13px; font-weight: bold; margin-top: 12px; display: block; width: 100%; }
    .info-sub { font-size: 10px; color: #888; }
</style>
"""))

user_db = {"admin": "12345"}
main_display = widgets.Output()

def get_map_view():
    center_pos = [10.7769, 106.7009]
    m = Map(center=center_pos, zoom=14, basemap=basemaps.CartoDB.DarkMatter, zoom_control=False, attribution_control=False)
    m.layout.height = '340px'

    user_dot = CircleMarker(location=center_pos, radius=6, color="#00d2d3", fill_color="#00d2d3", fill_opacity=1)
    user_halo = CircleMarker(location=center_pos, radius=12, color="#00d2d3", fill_opacity=0.15, weight=1)

    marker_end = Marker(location=[10.785, 106.712], draggable=True)
    dest_dot = CircleMarker(location=marker_end.location, radius=7, color="#ff7675", fill_color="#ff7675", fill_opacity=1)
    dest_halo = CircleMarker(location=marker_end.location, radius=14, color="#ff7675", fill_opacity=0.2, weight=1)

    path = AntPath(locations=[user_dot.location, marker_end.location], color="#00d2d3", pulse_color="#ffffff", weight=3, delay=500)

    m.add_layer(user_halo); m.add_layer(user_dot); m.add_layer(path)
    m.add_layer(dest_halo); m.add_layer(dest_dot); m.add_layer(marker_end)
    marker_end.opacity = 0.0

    vehicle = widgets.Dropdown(
        options=[('XE MÁY (Tiết kiệm & Nhanh chóng)', 5000),
                 ('XE Ô TÔ 4 CHỖ (Thoải mái & Riêng tư)', 15000),
                 ('XE Ô TÔ 6 CHỖ (Rộng rãi cho Gia đình)', 25000)],
        value=5000, layout={'width': '100%'}
    )
    trip_type = widgets.Dropdown(
        options=[('Chuyến đi Một chiều (Standard)', 1),
                 ('Chuyến đi Khứ hồi (Ưu đãi lượt về)', 1.8)],
        value=1, layout={'width': '100%'}
    )
    info_display = widgets.Output()

    def update_app_state(*args):
        dest_dot.location = marker_end.location
        dest_halo.location = marker_end.location
        path.locations = [user_dot.location, marker_end.location]
        p1, p2 = user_dot.location, marker_end.location
        dist = ((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2)**0.5 * 111
        total_price = dist * vehicle.value * trip_type.value
        with info_display:
            clear_output()
            display(HTML(f"""
            <div class="price-panel">
                <span class="info-sub">KHOẢNG CÁCH DỰ KIẾN:</span>
                <span style="font-size: 16px; font-weight: bold; float: right; color:white;">{dist:.2f} KM</span>
                <div style="margin-top: 5px;">
                    <span class="info-sub">TỔNG CHI PHÍ DỰ KIẾN:</span>
                    <span style="font-size: 18px; font-weight: bold; color: #fab1a0; float: right;">{total_price:,.0f} VND</span>
                </div>
            </div>
            """))

    marker_end.observe(update_app_state, names='location')
    vehicle.observe(update_app_state, names='value')
    trip_type.observe(update_app_state, names='value')
    m.on_interaction(lambda **kwargs: setattr(marker_end, 'location', kwargs.get('coordinates')) if kwargs.get('type') == 'click' else None)

    # POP-UP LOGIC
    popup_out = widgets.Output()

    def on_confirm_click(b):
        with popup_out:
            clear_output()
            display(HTML("""
                <div style="position: absolute; top: 50%; left: 50%; transform: translate(-50%, -50%);
                            background: white; padding: 20px; border-radius: 15px; border: 2px solid #00d2d3;
                            text-align: center; z-index: 1000; box-shadow: 0 0 20px rgba(0,0,0,0.5); width: 80%;">
                    <h3 style="color: #1a1a1a; margin: 0;">Thông báo</h3>
                    <p style="color: #333; margin: 10px 0;">Tài xế của bạn đang tới!</p>
                    <button onclick="this.parentElement.style.display='none'" style="background: #00d2d3; border: none; padding: 5px 15px; border-radius: 10px; font-weight: bold;">Đóng</button>
                </div>
            """))

    btn_confirm = widgets.Button(description="ĐẶT ĐƠN TÀI XẾ").add_class("order-btn")
    btn_confirm.on_click(on_confirm_click)

    btn_logout = widgets.Button(description="Đăng xuất tài khoản").add_class("secondary-btn")
    btn_logout.on_click(lambda x: show_login())

    ui_content = widgets.VBox([
        m,
        widgets.HTML('<div style="text-align:center; font-size:10px; color:#555; margin-top:5px">KÉO CHẤM ĐỎ HOẶC CHẠM BẢN ĐỒ</div>'),
        widgets.HTML('<div class="widget-label-custom">PHƯƠNG TIỆN DI CHUYỂN</div>'), vehicle,
        widgets.HTML('<div class="widget-label-custom">CHẾ ĐỘ NHIỆM VỤ</div>'), trip_type,
        info_display, btn_confirm, btn_logout, popup_out
    ]).add_class("app-content")

    update_app_state()
    return widgets.VBox([widgets.HTML('<div class="app-header">GO BIKE PRO</div>'), ui_content])

def show_login(b=None):
    with main_display:
        clear_output()
        u_login = widgets.Text(placeholder='TÊN NGƯỜI DÙNG', layout={'width': '100%'})
        p_login = widgets.Password(placeholder='MẬT KHẨU', layout={'width': '100%'})
        u_login.add_class("widget-text"); p_login.add_class("widget-password")
        btn_login = widgets.Button(description="ĐĂNG NHẬP").add_class("primary-btn")
        btn_to_reg = widgets.Button(description="Chưa có tài khoản? Đăng ký").add_class("secondary-btn")

        def handle_login(b):
            if u_login.value in user_db and user_db[u_login.value] == p_login.value:
                show_main_app()
            else:
                u_login.value = ""; u_login.placeholder = "Vui lòng nhập đúng thông tin!"

        btn_login.on_click(handle_login)
        btn_to_reg.on_click(show_register)
        display(widgets.VBox([widgets.HTML('<div class="logo-text">GO BIKE</div>'), u_login, p_login, btn_login, btn_to_reg]).add_class("screen-content"))

def show_register(b=None):
    with main_display:
        clear_output()
        u_reg = widgets.Text(placeholder='TÊN NGƯỜI DÙNG MỚI', layout={'width': '100%'})
        p_reg = widgets.Password(placeholder='MẬT KHẨU MỚI', layout={'width': '100%'})
        p_confirm = widgets.Password(placeholder='XÁC NHẬN MẬT KHẨU', layout={'width': '100%'})
        u_reg.add_class("widget-text"); p_reg.add_class("widget-password"); p_confirm.add_class("widget-password")
        btn_do_reg = widgets.Button(description="TẠO TÀI KHOẢN").add_class("primary-btn")
        btn_to_log = widgets.Button(description="Đã có tài khoản? Đăng nhập").add_class("secondary-btn")

        def handle_register(b):
            if u_reg.value and p_reg.value == p_confirm.value:
                user_db[u_reg.value] = p_reg.value
                show_login()
            else: u_reg.placeholder = "KIỂM TRA LẠI NHÉ!"

        btn_do_reg.on_click(handle_register)
        btn_to_log.on_click(show_login)
        display(widgets.VBox([widgets.HTML('<div class="logo-text" style="font-size: 30px;">ĐĂNG KÝ</div>'), u_reg, p_reg, p_confirm, btn_do_reg, btn_to_log]).add_class("screen-content"))

def show_main_app():
    with main_display:
        clear_output()
        display(get_map_view())

app_container = widgets.Box([main_display]).add_class("phone-frame")
display(app_container)
show_login()


Box(children=(Output(),), _dom_classes=('phone-frame',))